# Kaggle用: OEIS A007764 の自己回避路カウント

このノートブックは、元のコードをKaggleの標準Python環境で実行できるように整理したものです。

- 計算カーネルはKaggle上でgcc/ccからビルドするネイティブCです。
- CRTの各素数はCPUプロセスで独立に計算します。現在のカーネルはGPU計算ではないため、T4の有無に依存しません。
- `RUN_FULL = True` にすると `a(28)` の長時間計算を開始します。まずは `False` のまま Run All で環境確認できます。

In [ ]:
# 1. Kaggleランタイム確認
import multiprocessing
import os
import platform
import shutil
import subprocess

print(f"Python: {platform.python_version()}")
print(f"CPU workers visible: {os.cpu_count()}")
print(f"gcc: {shutil.which('gcc') or 'not found'}")
print(f"cc: {shutil.which('cc') or 'not found'}")

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    probe = subprocess.run(
        [nvidia_smi, '--query-gpu=name,memory.total', '--format=csv,noheader'],
        text=True,
        capture_output=True,
        check=False,
    )
    print('Visible GPUs:')
    print(probe.stdout.strip() or probe.stderr.strip() or 'none')
else:
    print('nvidia-smi: not found (GPU is not required by this kernel)')

In [ ]:
"""Kaggle-ready native multi-process engine for A007764 (n=28).

Standalone Python script that can be pasted into or executed from a Kaggle
Notebook.  The DP kernel is native C; multiple CRT primes are evaluated in
parallel on the Kaggle CPU workers.  CUDA is reported when available, but the
current kernel does not claim GPU acceleration.
"""

from __future__ import annotations
import concurrent.futures
import ctypes
import math
import multiprocessing
import os
import shutil
import subprocess
import sys
import tempfile
import time
from typing import Dict, List, Tuple

# Add math/src to path if present
current_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
src_dir = os.path.join(current_dir, "math", "src")
if os.path.exists(src_dir) and src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# OEIS A007764 Ground Truth for verification
KNOWN_A007764: Dict[int, int] = {
    1: 2,
    2: 12,
    3: 184,
    4: 8512,
    5: 1262816,
    6: 575780564,
    7: 789360053252,
    8: 3266598486981642,
    9: 41044208702632496804,
    10: 1568758030464750013214100,
    11: 182413291514248049241470885236,
    12: 64528039343270018963357185158482118,
}

# 64-bit Prime Pool for CRT (each prime < 2^62 for safe int64 modular arithmetic)
CRT_PRIMES_62BIT: List[int] = [
    4611686018427387847, 4611686018427387823, 4611686018427387799,
    4611686018427387751, 4611686018427387739, 4611686018427387709,
    4611686018427387687, 4611686018427387679, 4611686018427387653,
    4611686018427387641, 4611686018427387627, 4611686018427387593,
]

# 32-bit Prime Pool for fast 32-bit GPU/CPU ALU
CRT_PRIMES_32BIT: List[int] = [
    4294967291, 4294967279, 4294967231, 4294967197,
    4294967189, 4294967167, 4294967143, 4294967111,
    4294967087, 4294967029, 4294967011, 4294966981,
    4294966969, 4294966961, 4294966943, 4294966909,
]

# C source code for the high-performance bitboard DP kernel
C_DP_SOURCE = r"""
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>

typedef uint64_t u64;
typedef uint32_t u32;

#define EMPTY 0u
#define OPEN  1u
#define CLOSE 2u
#define MARK  3u

static inline unsigned getsym(u64 s, int k) { return (unsigned)((s >> (2 * k)) & 3u); }
static inline u64 setsym(u64 s, int k, unsigned v) {
    return (s & ~(3ULL << (2 * k))) | ((u64)v << (2 * k));
}

static inline int partner_open(u64 s, int k, int W) {
    int depth = 0;
    for (int t = k + 1; t < W; t++) {
        unsigned c = getsym(s, t);
        if (c == OPEN) depth++;
        else if (c == CLOSE) { if (!depth) return t; depth--; }
    }
    return -1;
}

static inline int partner_close(u64 s, int k) {
    int depth = 0;
    for (int t = k - 1; t >= 0; t--) {
        unsigned c = getsym(s, t);
        if (c == CLOSE) depth++;
        else if (c == OPEN) { if (!depth) return t; depth--; }
    }
    return -1;
}

static inline int partner(u64 s, int k, int W) {
    return getsym(s, k) == OPEN ? partner_open(s, k, W) : partner_close(s, k);
}

static const u64 EMPTY_KEY = ~0ULL;

typedef struct { u64 key, val; } Ent;

typedef struct {
    Ent *e;
    u32 *occ;
    size_t cap, mask, size;
} Table;

static void tab_alloc(Table *t, size_t cap) {
    t->cap = cap; t->mask = cap - 1; t->size = 0;
    t->e = (Ent *)malloc(cap * sizeof(Ent));
    t->occ = (u32 *)malloc((cap * 7 / 10 + 16) * sizeof(u32));
    if (!t->e || !t->occ) { fprintf(stderr, "Allocation failed (cap=%zu)\n", cap); exit(1); }
    for (size_t i = 0; i < cap; i++) t->e[i].key = EMPTY_KEY;
}

static void tab_free(Table *t) {
    if (t->e) free(t->e);
    if (t->occ) free(t->occ);
    t->e = NULL; t->occ = NULL;
}

static inline void tab_clear(Table *t) {
    for (size_t i = 0; i < t->size; i++) t->e[t->occ[i]].key = EMPTY_KEY;
    t->size = 0;
}

static inline u64 mix(u64 x) {
    x ^= x >> 33; x *= 0xff51afd7ed558ccdULL;
    x ^= x >> 33; x *= 0xc4ceb9fe1a85ec53ULL;
    x ^= x >> 33; return x;
}

static void tab_grow(Table *t) {
    Table nt; tab_alloc(&nt, t->cap * 2);
    for (size_t e = 0; e < t->size; e++) {
        Ent it = t->e[t->occ[e]];
        size_t i = mix(it.key) & nt.mask;
        while (nt.e[i].key != EMPTY_KEY) i = (i + 1) & nt.mask;
        nt.e[i] = it; nt.occ[nt.size++] = (u32)i;
    }
    free(t->e); free(t->occ);
    *t = nt;
}

static inline void tab_add(Table *t, u64 k, u64 v, u64 p) {
    size_t i = mix(k) & t->mask;
    for (;;) {
        u64 cur = t->e[i].key;
        if (cur == k) {
            u64 s = t->e[i].val + v;
            if (s >= p) s -= p;
            t->e[i].val = s;
            return;
        }
        if (cur == EMPTY_KEY) break;
        i = (i + 1) & t->mask;
    }
    t->e[i].key = k; t->e[i].val = v; t->occ[t->size++] = (u32)i;
    if ((t->size + 1) * 10 >= t->cap * 7) tab_grow(t);
}

#ifdef _WIN32
__declspec(dllexport)
#endif
u64 compute_an_mod_p(int n, u64 p, size_t init_cap_log2) {
    int C = n + 1, W = C + 1;
    if (W > 32) return 0;
    u64 fullmask = (W == 32) ? ~0ULL : ((1ULL << (2 * W)) - 1);

    size_t cap = (size_t)1 << (init_cap_log2 > 0 ? init_cap_log2 : 16);
    Table A, B, *cur = &A, *nxt = &B;
    tab_alloc(&A, cap);
    tab_alloc(&B, cap);
    tab_add(cur, 0ULL, 1ULL, p);

    for (int i = 0; i < C; i++) {
        for (int j = 0; j < C; j++) {
            int is_start = (i == 0 && j == 0);
            int is_end   = (i == C - 1 && j == C - 1);
            int can_down = (i < C - 1), can_right = (j < C - 1);
            tab_clear(nxt);
            size_t m = cur->size;
            for (size_t e = 0; e < m; e++) {
                size_t idx = cur->occ[e];
                u64 st = cur->e[idx].key, v = cur->e[idx].val;
                unsigned L = getsym(st, j), U = getsym(st, j + 1);
                u64 base = st & ~(15ULL << (2 * j));

                if (is_start) {
                    if (can_down)  tab_add(nxt, base | ((u64)MARK << (2 * j)), v, p);
                    if (can_right) tab_add(nxt, base | ((u64)MARK << (2 * j + 2)), v, p);
                } else if (is_end) {
                    if ((L == MARK && U == EMPTY) || (U == MARK && L == EMPTY))
                        tab_add(nxt, base, v, p);
                } else if (L == EMPTY && U == EMPTY) {
                    tab_add(nxt, base, v, p);
                    if (can_down && can_right)
                        tab_add(nxt, base | ((u64)OPEN << (2 * j)) | ((u64)CLOSE << (2 * j + 2)), v, p);
                } else if (U == EMPTY) {
                    if (can_down)  tab_add(nxt, base | ((u64)L << (2 * j)), v, p);
                    if (can_right) tab_add(nxt, base | ((u64)L << (2 * j + 2)), v, p);
                } else if (L == EMPTY) {
                    if (can_down)  tab_add(nxt, base | ((u64)U << (2 * j)), v, p);
                    if (can_right) tab_add(nxt, base | ((u64)U << (2 * j + 2)), v, p);
                } else if (L == OPEN && U == CLOSE) {
                    /* cycle exclusion */
                } else if (L == MARK) {
                    int q = partner(st, j + 1, W);
                    tab_add(nxt, setsym(base, q, MARK), v, p);
                } else if (U == MARK) {
                    int a = partner(st, j, W);
                    tab_add(nxt, setsym(base, a, MARK), v, p);
                } else {
                    int a = partner(st, j, W), b = partner(st, j + 1, W);
                    int lo = a < b ? a : b, hi = a < b ? b : a;
                    tab_add(nxt, setsym(setsym(base, lo, OPEN), hi, CLOSE), v, p);
                }
            }
            Table *t = cur; cur = nxt; nxt = t;
        }
        tab_clear(nxt);
        size_t m = cur->size;
        for (size_t e = 0; e < m; e++) {
            size_t idx = cur->occ[e];
            u64 st = cur->e[idx].key;
            if (getsym(st, C) != EMPTY) continue;
            tab_add(nxt, (st << 2) & fullmask, cur->e[idx].val, p);
        }
        { Table *t = cur; cur = nxt; nxt = t; }
    }

    u64 ans = 0;
    for (size_t e = 0; e < cur->size; e++) {
        if (cur->e[cur->occ[e]].key == 0ULL) {
            ans = cur->e[cur->occ[e]].val;
            break;
        }
    }
    tab_free(&A);
    tab_free(&B);
    return ans;
}
"""


def compile_c_engine() -> ctypes.CDLL | None:
    """Compile the optimized C bitboard DP kernel into an isolated shared library.

    A unique build directory is required because Kaggle workers compile/load the
    engine concurrently.  Sharing one /tmp/dp_engine.c and one .so causes
    occasional truncated libraries and loader errors.
    """
    lib_name = "libdp_engine.so" if sys.platform != "win32" else "libdp_engine.dll"
    compiler = shutil.which("gcc") or shutil.which("cc") or shutil.which("clang")
    if compiler is None:
        print("[Warning] gcc/cc/clang was not found; using the Python fallback engine.")
        return None

    tmp_dir = tempfile.mkdtemp(prefix=f"a007764_dp_{os.getpid()}_")
    c_path = os.path.join(tmp_dir, "dp_engine.c")
    lib_path = os.path.join(tmp_dir, lib_name)

    with open(c_path, "w") as f:
        f.write(C_DP_SOURCE)

    if sys.platform == "win32":
        cmd = [compiler, "-O3", "-shared", "-o", lib_path, c_path]
    else:
        cmd = [compiler, "-O3", "-fPIC", "-shared", "-o", lib_path, c_path]

    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    except Exception as e:
        print(f"[Warning] Native compilation failed: {e}. Using the Python fallback engine.")
        return None

    dll = ctypes.CDLL(lib_path)
    dll.compute_an_mod_p.argtypes = [ctypes.c_int, ctypes.c_uint64, ctypes.c_size_t]
    dll.compute_an_mod_p.restype = ctypes.c_uint64
    return dll


def _get_slot(bb: int, k: int) -> int:
    return (bb >> (2 * k)) & 3


def _set_slots_2(bb: int, k: int, v0: int, v1: int) -> int:
    mask = ~(15 << (2 * k)) & 0xFFFFFFFFFFFFFFFF
    return (bb & mask) | ((v0 & 3) << (2 * k)) | ((v1 & 3) << (2 * k + 2))


def _find_partner(bb: int, k: int, width: int) -> int:
    symbol = _get_slot(bb, k)
    depth = 0
    if symbol == 1:
        for t in range(k + 1, width):
            other = _get_slot(bb, t)
            if other == 1:
                depth += 1
            elif other == 2:
                if depth == 0:
                    return t
                depth -= 1
    elif symbol == 2:
        for t in range(k - 1, -1, -1):
            other = _get_slot(bb, t)
            if other == 2:
                depth += 1
            elif other == 1:
                if depth == 0:
                    return t
                depth -= 1
    raise AssertionError(f"Unmatched bracket at slot {k} in bitboard {bb:#x}")


def _run_python_dp(n: int, p: int) -> int:
    """Small-instance fallback used only when a C compiler is unavailable."""
    C = n + 1
    W = C + 1
    layer = {0: 1}
    for i in range(C):
        for j in range(C):
            is_start = i == 0 and j == 0
            is_end = i == C - 1 and j == C - 1
            can_down = i < C - 1
            can_right = j < C - 1
            nxt = {}

            def add(bb: int, value: int) -> None:
                nxt[bb] = (nxt.get(bb, 0) + value) % p

            for bb, value in layer.items():
                pair = (bb >> (2 * j)) & 15
                left, up = pair & 3, (pair >> 2) & 3

                def emit(down: int, right: int) -> None:
                    if down and not can_down:
                        return
                    if right and not can_right:
                        return
                    add(_set_slots_2(bb, j, down, right), value)

                if is_start:
                    emit(3, 0)
                    emit(0, 3)
                elif is_end:
                    if (left == 3) != (up == 3) and (left == 0 or up == 0):
                        add(_set_slots_2(bb, j, 0, 0), value)
                elif left == 0 and up == 0:
                    emit(0, 0)
                    if can_down and can_right:
                        emit(1, 2)
                elif up == 0:
                    emit(left, 0)
                    emit(0, left)
                elif left == 0:
                    emit(up, 0)
                    emit(0, up)
                elif left == 1 and up == 2:
                    continue
                elif left == 3:
                    q = _find_partner(bb, j + 1, W)
                    nb = _set_slots_2(bb, j, 0, 0)
                    add((nb & ~(3 << (2 * q))) | (3 << (2 * q)), value)
                elif up == 3:
                    q = _find_partner(bb, j, W)
                    nb = _set_slots_2(bb, j, 0, 0)
                    add((nb & ~(3 << (2 * q))) | (3 << (2 * q)), value)
                else:
                    p1 = _find_partner(bb, j, W)
                    p2 = _find_partner(bb, j + 1, W)
                    lo, hi = sorted((p1, p2))
                    nb = _set_slots_2(bb, j, 0, 0)
                    nb = (nb & ~(3 << (2 * lo))) | (1 << (2 * lo))
                    nb = (nb & ~(3 << (2 * hi))) | (2 << (2 * hi))
                    add(nb, value)
            layer = nxt

        shifted = {}
        for bb, value in layer.items():
            if _get_slot(bb, C) == 0:
                nb = (bb & ((1 << (2 * C)) - 1)) << 2
                shifted[nb] = (shifted.get(nb, 0) + value) % p
        layer = shifted
    return layer.get(0, 0)


def extended_gcd(a: int, b: int) -> Tuple[int, int, int]:
    if a == 0:
        return b, 0, 1
    gcd, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return gcd, x, y


def crt_reconstruct(residues: List[int], primes: List[int]) -> Tuple[int, int]:
    total: int = 0
    N: int = 1
    for p in primes:
        N *= p
    for r, p in zip(residues, primes):
        n_i = N // p
        _, inv, _ = extended_gcd(n_i, p)
        inv = inv % p
        total = (total + r * n_i * inv) % N
    return total, N


_WORKER_DLL: ctypes.CDLL | None = None


def _init_worker() -> None:
    global _WORKER_DLL
    _WORKER_DLL = compile_c_engine()


def _worker_task(n: int, p: int, init_cap_log2: int) -> Tuple[int, int, float]:
    """Worker task executing bitboard DP modulo p."""
    t0 = time.time()
    dll = _WORKER_DLL
    if dll:
        ans = dll.compute_an_mod_p(n, p, init_cap_log2)
    else:
        if n > 14:
            raise RuntimeError("A native C compiler is required for n > 14 on Kaggle.")
        ans = _run_python_dp(n, p)
    elapsed = time.time() - t0
    return p, ans, elapsed


def solve_a28_parallel(n: int, max_workers: int | None = None) -> Tuple[int, float, List[Tuple[int, int, float]]]:
    """Solve exact a(n) with independent CRT-prime CPU workers."""
    # Five 62-bit primes provide a conservative ~310-bit working modulus for n=28.
    # For smaller verification runs, select only the primes needed by the estimate.
    est_bits = max(64, int(n * 8.5) + 30)
    primes_used: List[int] = []
    prod = 1
    for p in CRT_PRIMES_62BIT:
        primes_used.append(p)
        prod *= p
        if prod.bit_length() > est_bits:
            break

    print(f"[*] Solving a({n}) using {len(primes_used)} 62-bit primes ({prod.bit_length()} bits total capacity)...")

    work_items = [(n, p, 18) for p in primes_used]
    results: List[Tuple[int, int, float]] = []

    t0 = time.time()
    if max_workers is None:
        max_workers = min(len(primes_used), os.cpu_count() or 1)
    max_workers = max(1, min(max_workers, len(primes_used)))
    with concurrent.futures.ProcessPoolExecutor(
        max_workers=max_workers,
        initializer=_init_worker,
    ) as executor:
        futures = [executor.submit(_worker_task, w[0], w[1], w[2]) for w in work_items]
        for f in concurrent.futures.as_completed(futures):
            res = f.result()
            results.append(res)
            print(f"  [Prime Complete] p = {res[0]} -> Residue a({n}) mod p = {res[1]:>20d} (in {res[2]:.2f}s)")

    # Reconstruct CRT
    res_map = {p: r for p, r, _ in results}
    ordered_residues = [res_map[p] for p in primes_used]
    exact_val, modulus = crt_reconstruct(ordered_residues, primes_used)
    total_time = time.time() - t0

    return exact_val, total_time, results


# Compile once in the notebook process. Worker processes compile into their own
temporary directories through _init_worker(), so parallel startup is safe.
c_engine = compile_c_engine()
if c_engine is None:
    print("[!] Native engine unavailable: only small verification cases can run.")
else:
    print("[*] Native engine ready.")

## 2. 既知値による検証

小さい `n` の既知値を確認してから並列CRTを起動します。

In [ ]:
# 2. n=1..8 の既知値検証
print('=' * 72)
print('Known-value verification')
print('=' * 72)
for test_n in range(1, 9):
    p = CRT_PRIMES_62BIT[0]
    ans = c_engine.compute_an_mod_p(test_n, p, 10) if c_engine else _run_python_dp(test_n, p)
    expected = KNOWN_A007764[test_n] % p
    assert ans == expected, f'Mismatch at n={test_n}: {ans} != {expected}'
    print(f'[PASS] n={test_n}: {KNOWN_A007764[test_n]}')
print('Known-value verification complete.')

In [ ]:
# 3. ProcessPool + CRT のスモークテスト
smoke_value, smoke_time, smoke_results = solve_a28_parallel(n=8, max_workers=2)
assert smoke_value == KNOWN_A007764[8], f'Parallel CRT mismatch: {smoke_value}'
print(f'[PASS] parallel a(8)={smoke_value} in {smoke_time:.2f}s')

## 4. `a(28)` 本計算

本計算はメモリ・時間を消費します。Kaggleで実行する場合のみ、次セルの `RUN_FULL` を `True` に変更してください。

In [ ]:
# 4. Full a(28) run (opt-in)
RUN_FULL = False

if RUN_FULL:
    a28_value, a28_seconds, a28_results = solve_a28_parallel(
        n=28,
        max_workers=min(4, os.cpu_count() or 1),
    )
    print(f'Exact a(28) = {a28_value}')
    print(f'Bit length  = {a28_value.bit_length()}')
    print(f'Digits      = {len(str(a28_value))}')
    print(f'Wall time   = {a28_seconds / 60:.2f} minutes')
else:
    print('Full run skipped. Set RUN_FULL = True to compute a(28).')